# Cardiac Segmentation — Mamba Integration Study
**Q1 Paper Training Notebook for Google Colab Pro**

Trains 9 base models + 10 Mamba-enhanced variants (Mamba/Mamba2/VMamba) on CAMUS dataset.

## Requirements
- Colab Pro (A100/V100/T4 GPU)
- CAMUS dataset in Google Drive at `CAMUS_public/database_nifti/`

## 1. Setup Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install PyTorch with CUDA 12.1
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
# Install mamba-ssm + causal-conv1d (CUDA kernels)
!pip uninstall -y mamba-ssm causal-conv1d 2>/dev/null
!pip install causal-conv1d mamba-ssm --no-build-isolation -q

In [ ]:
# Clone repo (pulls latest fixes)
!rm -rf /content/mamba
!git clone https://github.com/Seraf00/mamba.git /content/mamba
%cd /content/mamba

In [ ]:
# Install remaining dependencies
!pip install uv -q
!uv pip install -r requirements.txt

In [ ]:
# Suppress noisy TF/Triton warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TRITON_F32_DEFAULT'] = 'ieee'

## 2. Copy CAMUS Dataset

In [ ]:
import os

source_path = '/content/drive/MyDrive/CAMUS_public/database_nifti/'
destination_path = '/content/mamba/data/CAMUS'
os.makedirs(destination_path, exist_ok=True)

# Check if already copied (skip if resuming session)
if len(os.listdir(destination_path)) < 100:
    print(f'Copying CAMUS dataset... (5-10 min)')
    !rsync -ah --info=progress2 "{source_path}" "{destination_path}"
    print('Done!')
else:
    print(f'CAMUS dataset already present ({len(os.listdir(destination_path))} items)')

## 3. Verify Setup

In [ ]:
!python scripts/check_mamba_setup.py

In [ ]:
# Enable TensorBoard for live monitoring
%load_ext tensorboard

## 4. Helper: Save Results to Drive
Results on Colab are ephemeral — this syncs them to Google Drive after each training run.

In [ ]:
import shutil

DRIVE_RESULTS = '/content/drive/MyDrive/Paper1/results'

def sync_results_to_drive(local_dir, label=''):
    """Copy results to Google Drive for persistence."""
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    dest = os.path.join(DRIVE_RESULTS, os.path.basename(local_dir))
    !rsync -ah --info=progress2 "{local_dir}/" "{dest}/"
    print(f'Results synced to {dest}')

---
## 5. Train Base Models
These are the reference models (no Mamba). ~1-2 hours total on A100.

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 16 \
    --early_stopping 20 \
    --num_workers 4 \
    --base_only \
    --mixed_precision \
    --output_dir /content/results/base_models/

sync_results_to_drive('/content/results/base_models')

## 6. Train Mamba-Enhanced Models (Mamba variant)
MambaBlock uses CUDA-optimized selective scan. ~1h per model on A100.

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 8 \
    --early_stopping 20 \
    --num_workers 4 \
    --mamba_only \
    --mamba_variants mamba \
    --mixed_precision \
    --output_dir /content/results/mamba_models/

sync_results_to_drive('/content/results/mamba_models')

## 7. Train Mamba2-Enhanced Models
Mamba2 uses SSD algorithm. Falls back to PyTorch if Triton kernel fails. ~1.5h per model.

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 8 \
    --early_stopping 20 \
    --num_workers 4 \
    --mamba_only \
    --mamba_variants mamba2 \
    --mixed_precision \
    --output_dir /content/results/mamba2_models/

sync_results_to_drive('/content/results/mamba2_models')

## 8. Train VMamba-Enhanced Models
VMamba uses 4-directional cross-scan with selective_scan_fn CUDA kernels.

**Important**: AMP dtype fix ensures CUDA kernels work. Without fix, falls back to Python (~750h/epoch).
With fix + CUDA kernels: ~2-3h per model.

In [ ]:
!python scripts/train_all_models.py \
    --data_dir /content/mamba/data/CAMUS/ \
    --epochs 100 \
    --batch_size 4 \
    --early_stopping 20 \
    --num_workers 4 \
    --mamba_only \
    --mamba_variants vmamba \
    --mixed_precision \
    --output_dir /content/results/vmamba_models/

sync_results_to_drive('/content/results/vmamba_models')

## 9. Monitor Training (TensorBoard)

In [ ]:
# Launch TensorBoard to monitor live training curves
%tensorboard --logdir /content/results/

## 10. Evaluate All Models (after training)
Computes Dice, IoU, HD95, ASD + Biplane EF with pixel spacing.

In [ ]:
# Evaluate all trained models on test set
import glob

result_dirs = glob.glob('/content/results/*/benchmark_*')
for result_dir in sorted(result_dirs):
    print(f'\nEvaluating models in {result_dir}...')
    !python scripts/evaluate_all_models.py \
        --results_dir "{result_dir}" \
        --data_dir /content/mamba/data/CAMUS/ \
        --compute_ef \
        --split test

sync_results_to_drive('/content/results')

## 11. Final Sync

In [ ]:
# Sync everything to Drive one final time
!rsync -ah --info=progress2 /content/results/ "{DRIVE_RESULTS}/"
print(f'All results saved to Google Drive: {DRIVE_RESULTS}')